<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/05-agents/03-guardrails-and-budgets.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 护栏与预算

**目标：** 加入停止条件和成本/延迟预算；判断什么时候 pipeline 胜过 agent。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 环境准备

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（就是本仓库）带有共享的环境准备助手，并会拉入 `groq` 客户端，这是本 notebook 唯一需要的依赖。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领一个免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的 **钥匙图标** → **Add new secret** 添加它，名称必须严格写成 `GROQ_API_KEY`，粘贴值，然后打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整走查和选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），并返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 生产环境的问题

notebook 01 里的 demo agent 能跑。生产环境的问题不一样：**当它跑不动时，什么能让它停下来？** 一个没有边界的 agent 循环就是一笔没有边界的账单：一个总能再找到一件事要查的模型、一个招来无限重试的不稳定工具、一个从一开始就不可能完成的任务。这些都不是假设；它们都曾为某个人产生过四位数美元的 API 账单。

本 notebook 拿 notebook 01 的循环，逐个加上五个护栏，每个都配一个演示：

1. **max_turns** —— 我们本来就有；现在我们真的会撞上它。
2. **成本预算** —— 由每一轮真实的 `response.usage` 数字累加而来。
3. **墙上时钟超时。**
4. **工具级护栏** —— 对破坏性工具设一道确认门禁。
5. **循环检测** —— 同一个工具、同样的参数，连续两次就触发一次干预。

In [ ]:
import json
import re
import time

# 和 notebook 01 相同的假文件系统 + 计算器，多加几个文件，
# 让多步任务有东西可啃。
FS = {
    "data/q1.txt": "Q1 revenue: $12,400. Churn: 3.1%.",
    "data/q2.txt": "Q2 revenue: $13,900. Churn: 2.8%.",
    "data/q3.txt": "Q3 revenue: $15,200. Churn: 2.9%.",
    "data/q4.txt": "Q4 revenue: $16,800. Churn: 2.5%.",
    "notes/board.md": "Board wants a one-page revenue summary by Friday.",
    "notes/todo.md": "1. Summarize quarters. 2. Flag churn trend.",
}

def list_files():
    return "\n".join(sorted(FS)) if FS else "(no files)"

def read_file(path):
    if path not in FS:
        return f"Error: no file at '{path}'. Call list_files to see what exists."
    return FS[path]

def write_file(path, content):
    FS[path] = content
    return f"Wrote {len(content)} characters to {path}."

def delete_file(path):
    if path not in FS:
        return f"Error: no file at '{path}'."
    del FS[path]
    return f"Deleted {path}."

def calculator(expression):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Error: only numbers and + - * / ( ) are allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

IMPLS = {
    "list_files": list_files, "read_file": read_file, "write_file": write_file,
    "delete_file": delete_file, "calculator": calculator,
}

def call_tool(name, args, impls):
    """安全地分发一次工具调用。模型有时会编造一个函数并不接受的参数
    （例如 list_files(path=...)）—— 把它变成一段模型能读懂并从中恢复的错误字符串，
    绝不要变成搞死循环的异常。
    （notebook 02 的「能教学的错误」规则，用在分发层。）
    """
    fn = impls.get(name)
    if fn is None:
        return f"Error: unknown tool '{name}'."
    try:
        return str(fn(**args))
    except TypeError as e:
        return f"Error: bad arguments for {name}: {e}. Check the tool's schema and try again."

def tool(name, description, props, required):
    """按 Groq 使用的 OpenAI 兼容格式构造一个工具定义。"""
    return {"type": "function",
            "function": {"name": name, "description": description,
                         "parameters": {"type": "object", "properties": props,
                                        "required": required}}}

_str = lambda: {"type": "string"}
TOOLS = [
    tool("list_files", "List every file path in the workspace, one per line.", {}, []),
    tool("read_file", "Read one file's full contents. Returns text or an error.",
         {"path": _str()}, ["path"]),
    tool("write_file", "Create or overwrite a file. Destructive: replaces existing content.",
         {"path": _str(), "content": _str()}, ["path", "content"]),
    tool("delete_file", "Delete a file permanently. Destructive and irreversible.",
         {"path": _str()}, ["path"]),
    tool("calculator",
         "Evaluate an arithmetic expression using + - * / and parentheses. "
         "Use for all math. Returns the numeric result.",
         {"expression": _str()}, ["expression"]),
]


## 护栏 1：max_turns，以及真的撞上它

notebook 01 的循环早就有 `max_turns`；只是因为它从没触发过，因为 demo 任务太简单。这里再来一遍紧凑版的循环，返回一个结构化结果而不是光秃秃的字符串，这样调用方能知道这次运行是*怎么*结束的，这也是后面每个护栏都会沿用的习惯。然后我们给它超出回合预算的工作量，看它停下来而不是空转。

In [ ]:
def run_agent(task, tools, impls, max_turns=10):
    """notebook 01 的循环，紧凑版，返回一个结构化结果。"""
    messages = [{"role": "user", "content": task}]

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages,
        )
        choice = response.choices[0]
        msg = choice.message
        if choice.finish_reason != "tool_calls":
            return {"status": "done", "turns": turn, "text": msg.content or ""}

        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            out = call_tool(tc.function.name, args, impls)
            print(f"[turn {turn}] {tc.function.name}({tc.function.arguments[:70]}) -> {out.strip()[:70]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

    return {"status": "max_turns", "turns": max_turns,
            "text": f"Stopped at max_turns={max_turns} before finishing."}

# 工作量超出 3 轮能做完的量：六个文件，每个都要有自己的汇总文件。
result = run_agent(
    "For every file in the workspace, write a companion file named "
    "<original_path>.summary containing a one-line summary of it. "
    "Do the files one at a time, in alphabetical order.",
    TOOLS, IMPLS, max_turns=3,
)
print("\n", result["status"], "-", result["text"][:120])


跑一下，注意结果：正好 3 轮之后是 `status: max_turns`。任务没完成，而这正是要点。没有这个边界，这次运行会一直跑到模型需要的轮数（那倒没问题），但一次模型卡在反复重读同一个文件的运行也会如此（这就不行了）。这个边界把「可能无边界」变成「至多 N 次调用」，而结构化状态告诉调用方结果是部分的，而不是悄悄把截断的工作当成完成交出去。

不过回合上限是个粗糙的工具：回合不是美元。下一个护栏统计的才是真正花钱的东西。

## 护栏 2–5：带预算的循环

下面是完整的带护栏循环。还是那约 40 行，加四处新增，每一处都用一条 `guard N` 注释标出：

- **护栏 2（成本预算）：** 每个响应都带 `response.usage`；我们每一轮把 token 换算成美元，当花费越线时带着部分 trace 中止。这个护栏直接对应到你的账单。
- **护栏 3（墙上时钟超时）：** 在每一轮开头检查。它很粗糙（单次慢 API 调用可能冲过头），但能封住漂移。生产系统会在底下再加逐请求超时。
- **护栏 4（确认门禁）：** 被列为破坏性的工具，除非 `confirm(name, args)` 回调批准，否则不运行。默认是**拒绝**。在 notebook 里你可以传 `confirm=lambda name, args: input(f'{name}? [y/n] ') == 'y'`；在一个服务里，这就变成一个人工审批队列。
- **护栏 5（循环检测）：** 如果模型连续两次发出完全相同的工具调用，我们就注入一个 user 轮干预，告诉它换个做法或者报告失败。对一个确定性工具重试一模一样的调用是纯粹的浪费。

In [ ]:
# $/百万 token 的示意费率（Groq 免费档是 $0 —— 加上这个才能让成本护栏有点事做）。
PRICE_IN_PER_MTOK = 0.59
PRICE_OUT_PER_MTOK = 0.79

INTERVENTION = (
    "You already made that exact tool call and got the same result. Do not repeat it. "
    "Try a different approach, or report that you cannot complete the task and why."
)

def run_agent_guarded(task, tools, impls, max_turns=10,
                      max_cost_usd=0.05,        # 护栏 2
                      max_seconds=120,          # 护栏 3
                      confirm=None,             # 护栏 4（None = 拒绝所有破坏性调用）
                      destructive=("write_file", "delete_file")):
    messages = [{"role": "user", "content": task}]
    cost, started, last_call, trace = 0.0, time.monotonic(), None, []

    def stopped(status, turn, text=""):
        return {"status": status, "turns": turn, "cost_usd": round(cost, 4),
                "text": text, "partial_trace": trace[-6:]}

    for turn in range(1, max_turns + 1):
        # ---- 护栏 3：墙上时钟超时 ----
        if time.monotonic() - started > max_seconds:
            return stopped("timeout", turn - 1)

        response = client.chat.completions.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages,
        )
        choice = response.choices[0]
        msg = choice.message

        # ---- 护栏 2：由真实用量数字算出成本预算 ----
        cost += (response.usage.prompt_tokens * PRICE_IN_PER_MTOK
                 + response.usage.completion_tokens * PRICE_OUT_PER_MTOK) / 1_000_000
        if cost > max_cost_usd:
            return stopped("budget_exceeded", turn)

        if choice.finish_reason != "tool_calls":
            return stopped("done", turn, msg.content or "")

        messages.append(msg)
        repeat = False
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)

            # ---- 护栏 5：与上一次调用相同的工具 + 相同的参数 ----
            sig = (name, json.dumps(args, sort_keys=True))
            repeat = repeat or (sig == last_call)
            last_call = sig

            # ---- 护栏 4：破坏性工具上的确认门禁 ----
            if name in destructive:
                approved = confirm(name, args) if confirm else False
                if not approved:
                    out = (f"Denied: {name} requires user approval, which was not "
                           "granted. Do not retry it. Finish the task without this action, "
                           "or report what you would have done instead.")
                    print(f"[turn {turn}] {name} DENIED by confirmation gate")
                    trace.append(f"{name} DENIED")
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
                    continue

            out = call_tool(name, args, impls)
            print(f"[turn {turn}] {name}({tc.function.arguments[:70]}) -> {out.strip()[:70]}")
            trace.append(f"{name} -> {out.strip()[:60]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

        # ---- 护栏 5（续）：把干预作为 user 轮注入 ----
        if repeat:
            print(f"[turn {turn}] LOOP DETECTED -- injecting intervention")
            messages.append({"role": "user", "content": INTERVENTION})

    return stopped("max_turns", max_turns)


### 演示：护栏 2，成本预算

给它同一个超大任务，预算只有几分之一美分。跑一下，注意 `budget_exceeded` 状态、实际花费，以及 `partial_trace`：调用方拿到一张「*已经*做了什么」的收据，而不只是一个失败。

In [ ]:
result = run_agent_guarded(
    "For every file in the workspace, write a companion file named "
    "<original_path>.summary containing a one-line summary of it. "
    "Do the files one at a time.",
    TOOLS, IMPLS,
    max_cost_usd=0.01,   # 故意设得极小 —— 按当前价格大约撑 2-3 轮
)
print()
print(json.dumps({k: v for k, v in result.items() if k != "text"}, indent=2))


trace 让一件事变得可见：每一轮的成本随着运行推进而*上升*，因为每一轮都要重发整段历史（notebook 01 结尾的观察）。一个固定的逐轮估算会低估长运行；从 `response.usage` 累加才是诚实的度量。

### 演示：护栏 3，墙上时钟超时

同一个任务，成本预算给得宽松，但超时短于这次运行。检查发生在轮次之间，所以预期这次运行会跑完正在进行的那一轮再停下。注意 `timeout` 状态，以及 `turns` 很小。

In [ ]:
result = run_agent_guarded(
    "For every file in the workspace, write a companion file named "
    "<original_path>.summary containing a one-line summary of it. "
    "Do the files one at a time.",
    TOOLS, IMPLS,
    max_cost_usd=1.00,
    max_seconds=8,       # 一轮通常要几秒；两轮就塞不下了
)
print()
print(result["status"], "after", result["turns"], "turns,", f"${result['cost_usd']}")


### 演示：护栏 4，确认门禁

`confirm=None` 时每一个破坏性调用都被自动拒绝。下面这个任务*要求*一次写入，所以跑一下，看模型撞上门禁、读到拒绝，然后（因为拒绝消息告诉了它该怎么做）在它的回答里汇报汇总，而不是重试写入。（notebook 02 的「能教学的错误」规则，用在一则策略消息上。）

要在 Colab 里交互式批准，传
`confirm=lambda name, args: input(f"allow {name}? [y/N] ").strip().lower() == "y"`。

In [ ]:
result = run_agent_guarded(
    "Read the four quarterly files in data/ and write a file report.md with total "
    "annual revenue (use the calculator) and one sentence on the churn trend.",
    TOOLS, IMPLS,
    confirm=None,        # 自动拒绝：write_file 和 delete_file 会被挡住
)
print()
print(result["status"], "-", result["text"][:400])
print("\nreport.md written?", "report.md" in FS)


### 演示：护栏 5，循环检测

为了可靠地触发一次真正的重试循环，我们加一个做了手脚的工具：`fetch_report` 总是带着一个*听起来可重试*的错误失败（"temporarily unavailable, try again"）。这个措辞是诱饵，正是易抖动的上游 API 会产出的东西，而模型常常会上钩。跑一下，看这个序列：一模一样的调用、一模一样的调用、`LOOP DETECTED`、干预被注入，然后模型要么换个做法，要么干净地报告失败，而不是把剩下的轮次都烧在重试上。

In [ ]:
def fetch_report(name):
    return "Error: report service temporarily unavailable. Try again."

FLAKY_TOOLS = TOOLS + [
    tool("fetch_report",
         "Fetch a named report from the reporting service. Returns the report text.",
         {"name": _str()}, ["name"]),
]
FLAKY_IMPLS = {**IMPLS, "fetch_report": fetch_report}

result = run_agent_guarded(
    "Fetch the 'quarterly' report with fetch_report and summarize it in two sentences.",
    FLAKY_TOOLS, FLAKY_IMPLS,
    max_turns=6,
)
print()
print(result["status"], "-", result["text"][:400])


关于机制的一点说明：干预是作为 **user 轮**进去的，所以它成为模型据以推理的对话的一部分，和人类操作员会用的是同一个通道。我们的检测器刻意做得很简单（连续两次完全相同的调用）；生产版本还会抓 A→B→A→B 交替和近似重复的参数，而且会升级处置：干预一次，之后就中止。

## 停止条件表

| 条件 | 它防范什么 | 触发时怎么做 |
|---|---|---|
| `max_turns` | 失控的计划、无限循环 | 大声失败：返回 status + trace，绝不悄悄截断 |
| 成本预算 | 无边界账单；冗长上下文的爆表 | 返回部分结果 + 实际花费；如果频繁触发就告警 |
| 墙上时钟超时 | 卡住的工具、缓慢漂移、上游卡死 | 大声失败；在底下配逐请求超时 |
| 确认门禁 | 不可逆动作：删除、发送、付款 | 问人；没人可问时默认拒绝 |
| 循环检测 | 对确定性失败做徒劳的重试 | 带引导干预一次；第二次犯规就中止 |

共同的主线是：**每一次触发都产出一个可读的结果**，一个状态、一个花费数字、一份部分 trace。一个带着收据大声失败的 agent 是可运维的；一个悄悄失败的 agent 是一场调试噩梦。

## 评测回来了：给*轨迹*打分，而不只是给答案

第 02 节装好了评测脊柱，第 04 节度量了 RAG 的*答案*。agent 还需要一样东西：对一个 agent 来说，一个看起来对的答案如果是靠一条疯狂的路径达成的，那仍然是 bug。它烧掉了预算、调用了错误的工具，或者以一种不会重现的方式走了运。所以你要给**轨迹**（工具调用的序列）打分，而不只是最终文本。

好消息是：数据你已经有了。`run_agent` 返回轮次和 trace；轨迹评测就是对这份 trace 做断言。下面是这个极小的 `(task, expected-tool-behavior)` 黄金集和一个检查器，即第 02 节黄金集在 agent 上的对应物。

In [ ]:
# 一次记录自身工具调用序列的运行，这样我们能给路径打分，而不只是文本。
def run_with_trace(task, tools, impls, max_turns=8):
    messages = [{"role": "user", "content": task}]
    calls = []  # 轨迹：agent 调用过的工具名的有序列表
    for _ in range(max_turns):
        resp = client.chat.completions.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages)
        msg = resp.choices[0].message
        if resp.choices[0].finish_reason != "tool_calls":
            return {"text": msg.content or "", "calls": calls}
        messages.append(msg)
        for tc in msg.tool_calls:
            calls.append(tc.function.name)
            out = call_tool(tc.function.name, json.loads(tc.function.arguments), impls)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return {"text": "[max_turns]", "calls": calls}

# 给轨迹用的黄金集：每个 case 断言的是工具路径的某件事，
# 不是答案的措辞。这就是第 02 节黄金集的想法用在 agent 上。
TRAJECTORY_CASES = [
    {"task": "What is 12400 + 13900? Use your tools.",
     # 必须用计算器；对一个纯数学问题必须不碰文件系统
     "check": lambda c: "calculator" in c and not any(x in c for x in
                        ("read_file", "write_file", "delete_file"))},
    {"task": "Summarize what's in data/q1.txt.",
     # 必须读它被问到的那个文件；不应该需要计算器
     "check": lambda c: "read_file" in c},
]

print(f"{'ok':4} {'calls'}")
passed = 0
for case in TRAJECTORY_CASES:
    calls = run_with_trace(case["task"], TOOLS, IMPLS)["calls"]
    ok = case["check"](calls)
    passed += ok
    print(f"{'PASS' if ok else 'FAIL':4} {calls}   <- {case['task'][:40]}")
print(f"\ntrajectory pass rate: {passed}/{len(TRAJECTORY_CASES)}")

## 什么时候 pipeline 胜过 agent

这是面试会追问的判断，也是整个这一节一直在铺垫的那个判断。你已经搭过三遍的这个循环之所以强大，恰恰是因为模型自己选择下一步，而这也正是它的代价：非确定性、逐步的 API 花费，以及上面那五个护栏。

如果你能事先把步骤写下来，**就写 pipeline**：

- **步骤事先已知？** 抓取 → 抽取 → 总结 → 存储是一个 pipeline。把顺序写成代码；在需要语言能力的步骤里调用模型。
- **每次运行形状都一样？** 如果第 1000 次运行应该看起来像第 1 次，确定性就是一种特性。pipeline 可测试（对每个阶段做断言）、可调试（逐阶段日志）、且便宜（每个模型形状的步骤一次模型调用，没有重发的历史）。
- **路径真的取决于中间结果吗？** 只有这时你才需要循环：调试（「读这个错误，决定下一步检查什么」）、开放式研究，以及那些文件/工具/下一个问题事先无法知晓的任务。
- **拿不准？** 先上 pipeline。把那个真的需要分支的步骤，以后再提升成一个小型有界 agent。反向迁移，也就是在生产环境里把一个不稳定的系统去 agent 化，要痛苦得多。

> **🚩 常见错误 ——** 已上线的「agent」里有很大一部分是披着风衣的 pipeline：模型在那些永远按同一顺序运行的步骤之间「做决定」，而团队为 pipeline 的行为付 agent 的代价（延迟、花费、非确定性、评测难度）。面试里，「我们搭了一个 pipeline，因为工作流是已知的」是比「我们搭了一个 agent」*更强*的回答。



## 练习

1. **优雅地退出预算。** 当成本预算触发时，不要立刻返回，而是发*一次*最终请求（不带工具、`max_tokens` 设小），让模型总结到目前为止的进展，并把这次请求的成本计入总额。把结果的有用程度与原始的 `partial_trace` 对比一下。
2. **可升级的循环检测。** 把护栏 5 扩展成也能抓 A→B→A→B 交替（保留最近四个签名），并在模型*在干预之后*仍重复时以状态 `"looping"` 中止。用 `fetch_report` 这个诱饵验证。
3. **按工具的预算。** 加一个 `tool_limits` dict（例如 `{"fetch_report": 2}`），某个工具的调用计数耗尽后，用一条能教模型的错误拒绝它。确认那个不稳定的运行现在即使没有循环检测器，也能以一个干净的失败报告结束。
4. **把它去 agent 化。** 拿护栏 4 的演示任务（读四个季度文件、算总额、汇报 churn 趋势），改写成一条直白的 pipeline：对 `FS` 一次循环、一次 `sum()`，以及*一次*模型调用来组织汇总的措辞。对比它和 agent 版本烧掉的 token 与墙上时钟时间，并写一句话说明什么时候 agent 版本仍然值得。
